# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OjaswiGautam/FlyrankAI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item's daily performance record, at grain report_date × client_hash_id × content_hash_id. Verified with a full duplicate-group count, not a limited preview: 0 duplicate groups, 0 duplicate rows, max group size = 1 across all 9,841,378 rows — the grain claim holds exactly, not just plausibly. PASS.

Time window: month=2026-03, spanning 2026-03-01 to 2026-03-31, with 31 distinct calendar days confirmed against the 31 expected days in March — no gaps in the middle of the month. PASS (31/31 days present). This is deliberately a mid-panel month, not _sample (June 2026, the sealed final month / natural outcome window for any label).

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import duckdb
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")

TABLE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Grain probe: full duplicate count, not a LIMIT-ed preview
grain_check = con.sql(f"""
    SELECT
        COUNT(*) FILTER (WHERE c > 1) AS duplicate_groups,
        COALESCE(SUM(c) FILTER (WHERE c > 1), 0) AS duplicate_rows,
        COALESCE(MAX(c), 0) AS max_group_size
    FROM (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
        FROM read_parquet('{TABLE}')
        GROUP BY report_date, client_hash_id, content_hash_id
    )
""").df()
print("Grain check (full count, no LIMIT):")
print(grain_check)
print("PASS" if grain_check["duplicate_groups"].iloc[0] == 0 else "FAIL")

# Window check: min/max PLUS distinct day count vs expected 31
window_check = con.sql(f"""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT report_date) AS distinct_days,
        COUNT(*) AS row_count
    FROM read_parquet('{TABLE}')
""").df()
print("\nDate span + row count for this partition:")
print(window_check)
print("PASS (31/31 days present)" if window_check["distinct_days"].iloc[0] == 31 else "FAIL (missing days)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check (full count, no LIMIT):
   duplicate_groups  duplicate_rows  max_group_size
0                 0             0.0               1
PASS

Date span + row count for this partition:
    min_date   max_date  distinct_days  row_count
0 2026-03-01 2026-03-31             31    9841378
PASS (31/31 days present)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Lane 3's clustering dataframe is built by joining dim_content onto an aggregated monthly slice of fact_content_daily_performance. Aggregation grain after joining: one row per content_hash_id × client_hash_id × month — the daily report_date grain is collapsed via aggregation (sum/mean over March) before clustering; dim_content's static fields attach once per content_hash_id.

Label: None is used for model fitting — Lane 3 is unsupervised clustering with no supervised target. Any cluster ID, archetype name, or recommended action produced later is a downstream interpretation output, not a label and never fed back in as a feature.

Caveat on the join, corrected from an earlier overstatement: a prior check showed 9,841,378 fact rows all matching a content_hash_id in dim_content — this only proves the fact→dim direction has no unmatched keys. It does not prove dim_content has no duplicate content_hash_id values (which would silently fan out fact rows while still showing "matched"), and it says nothing about dim_content rows with zero fact-side matches (expected — not every page had March activity). Both need a dedicated check in Section 3, not inferred from the match count alone.

# Field Inventory

## 1. Join Keys / Grain

### `content_hash_id`
- **Source:** both
- **Bucket:** join key / grain
- **Role / use:** joins the two tables; part of clustering grain
- **QA status:** —
- **Decision / reason:** not a feature

### `client_hash_id`
- **Source:** both
- **Bucket:** grain / split
- **Role / use:** part of grain; also used for client-holdout-style splits if needed later
- **QA status:** —
- **Decision / reason:** not a feature

### `report_date`
- **Source:** fact
- **Bucket:** grain (pre-aggregation)
- **Role / use:** collapsed away once aggregated to the month
- **QA status:** —
- **Decision / reason:** not a feature after aggregation

---

## 2. Identifier-only Fields

### `keyword_hash_id`, `url_hash_id`
- **Source:** dim_content
- **Bucket:** identifier-only
- **Role / use:** no clustering use identified
- **QA status:** —
- **Decision / reason:** excluded — identifier-only

---

## 3. Filters / QA Fields

### `month`
- **Source:** fact
- **Bucket:** filter / QA
- **Role / use:** confirms correct partition selected
- **QA status:** —
- **Decision / reason:** not a feature

### `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available`
- **Source:** fact
- **Bucket:** filter / QA
- **Role / use:** used with IS TRUE to select valid rows before clustering
- **QA status:** —
- **Decision / reason:** excluded — availability flag, not behavior

### `is_published`, `is_deleted`
- **Source:** dim_content
- **Bucket:** filter / QA
- **Role / use:** likely used to exclude deleted/unpublished pages before clustering
- **QA status:** needs null/value check
- **Decision / reason:** excluded — status flag, not behavior

---

## 4. Candidate Inputs

### `content_type`
- **Source:** dim_content
- **Bucket:** candidate input
- **Role / use:** 3 categories, verified 0% null, sums exactly to 519,606
- **QA status:** verified clean
- **Decision / reason:** candidate for final 5-feature frame

### `word_count`
- **Source:** dim_content
- **Bucket:** candidate input
- **Role / use:** content shape signal
- **QA status:** verified 34.2% null
- **Decision / reason:** candidate, requires has_word_count flag — cannot fillna(0) blindly

### `search_volume`, `competition`
- **Source:** dim_content
- **Bucket:** candidate input
- **Role / use:** search-context signals
- **QA status:** verified 27.4% null, identical rate — correlated missingness
- **Decision / reason:** candidate, requires shared has_search_data flag

### `main_intent`, `char_count`, `backlinks`, `category_count`, `competition_level`, `cpc`
- **Source:** dim_content
- **Bucket:** candidate input
- **Role / use:** possible behavioral/structural signals
- **QA status:** not yet null-checked
- **Decision / reason:** still open — QA needed in Section 3 before inclusion decision

### `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `sessions_organic/direct/referral/social/paid/ai`, `scroll_events`
- **Source:** fact
- **Bucket:** candidate inputs considered
- **Role / use:** behavioral signals aggregated over March
- **QA status:** not yet null-checked
- **Decision / reason:** narrowed down to the actual 5 chosen features in Section 3, not all used

---

## 5. Derived Intermediate Fields

### `content_created_date`, `content_updated_date`
- **Source:** dim_content
- **Bucket:** derived intermediate
- **Role / use:** used to compute content_age_days, not clustered on raw
- **QA status:** —
- **Decision / reason:** intermediate only, not a direct feature

---

## 6. Excluded Fields

### `gsc_sum_position`
- **Source:** fact
- **Bucket:** excluded
- **Role / use:** duplicate/proxy of gsc_avg_position; not comparable across content with different observed-day counts
- **QA status:** —
- **Decision / reason:** excluded — duplicate/proxy of another field

### `keyword_char_count`, `keyword_token_count`, `url_char_count`, `keyword_created_date`
- **Source:** dim_content
- **Bucket:** excluded
- **Role / use:** describe the keyword/URL, not page behavior
- **QA status:** —
- **Decision / reason:** excluded — out of scope for a behavioral archetype

### `provider_used`, `model_used`
- **Source:** dim_content
- **Bucket:** excluded
- **Role / use:** describes generation method, not performance
- **QA status:** —
- **Decision / reason:** excluded — out of scope

### `optimization_eligible_date`, `last_optimized_date`
- **Source:** dim_content
- **Bucket:** excluded
- **Role / use:** verified 91.26% null, identical rate; adjacent to a FlyRank workflow decision
- **QA status:** verified
- **Decision / reason:** excluded — high-null/unstable and workflow/optimization output risk (possible circularity)

### `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other`
- **Source:** fact
- **Bucket:** excluded individually
- **Role / use:** likely too sparse per-channel
- **QA status:** not yet null-checked per column
- **Decision / reason:** excluded individually; folded into one derived ai_sessions_total instead

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Grain (Section 1): 0 duplicate groups across 9,841,378 raw daily rows in month=2026-03 — exact count, not sampled.

Window (Section 1): 31/31 calendar days present, 2026-03-01 to 2026-03-31.

Join integrity: content_hash_id confirmed globally unique in dim_content (0 duplicate keys) — safe as a join key alone. Fact→dim: 100% matched, 0 unmatched. 188,169 dim_content rows have no March activity (expected).

Availability: gsc_data_available IS TRUE → 3,611,061 / 9,841,378 raw rows (36.7%). ga4_data_available IS TRUE → 413,966 rows (4.2%). Both-available overlap → 3.7%, confirming GA4 is the binding constraint.

Final 5-feature frame — grain: exactly one row per content_hash_id × client_hash_id, for content items with ≥1 day of real GSC data in March. GSC-unavailable days were filtered before aggregation (not after), so sums and the impression-weighted average reflect only real observed data, not zero-filled placeholder rows.

# Final Feature Set

## 1. `gsc_impressions`

- **Source:** fact, GSC-filtered
- **Aggregation:** SUM
- **Nulls before:** 0
- **After:** —

---

## 2. `gsc_avg_position`

- **Source:** fact, GSC-filtered
- **Aggregation:** Impression-weighted: `SUM(impressions × position) / SUM(impressions)`
- **Nulls before:** 0
- **After:** —

---

## 3. `gsc_clicks`

- **Source:** fact, GSC-filtered
- **Aggregation:** SUM
- **Nulls before:** 0
- **After:** —

---

## 4. `content_age_days`

- **Source:** derived from `dim_content.content_created_date`
- **Aggregation:** DATE_DIFF vs. window end (2026-03-31)
- **Nulls before:** 0 nulls, 0 negative, range 0–494 days
- **After:** —

---

## 5. `word_count`

- **Source:** dim_content
- **Aggregation:** as-is
- **Nulls before:** 55,315 / 176,738 = 31.30%
- **After:** 0 (median-imputed + `word_count_missing` flag)

Final population: 176,738 content items — the modeling universe for Lane 3's March feature frame, out of dim_content's ~519,606 total (i.e., ~34% of all content has usable March GSC coverage; this is a named limitation for Section 4, not hidden).

ga4_engaged_sessions excluded from the core frame — confirmed via two checks: sessions_organic is 100% GA4-gated (0% independent coverage), and GSC/GA4 both-available overlap is only 3.7%. Retained only for post-cluster profiling on its valid subset, never as a clustering input.

Five features, each with "knowable at the decision moment because…":



*   gsc_impressions — aggregate of already-observed March search performance
*   gsc_avg_position — aggregate of already-observed ranking data, correctly weighted by daily impression volume rather than averaged naively across days
*   gsc_clicks — aggregate of already-observed historical clicks
content_age_days — computed from a fixed, past fact (content_created_date), relative to the window end, not "today"
*   word_count (imputed + flagged) — a static property of the page as recorded in dim_content








## THE LEAK TRAP :

The finalized 5-feature frame (gsc_impressions, gsc_avg_position, gsc_clicks, content_age_days, word_count) produced an honest silhouette score of 0.2972, clearing the ≥0.25–0.30 target set in the w02 success-metric section.

As a diagnostic, cluster membership from that honest model was one-hot encoded (never ordinal — cluster IDs carry no real distance relationship) and appended to the feature space. Recomputing silhouette on the unchanged cluster assignment gave 0.3947; refitting K-Means on the leak-augmented matrix gave the same 0.3947 — expected, since a perfectly-encoded leak of the labels gives the refit nothing new to discover, so it converges to essentially the same partition.

Both diagnostics show a +0.0975 (~33% relative) jump over the honest baseline. This mirrors the leakage lesson from notebook 02 (trend_direction → is_declining_label): a feature derived from the model's own output makes the result look artificially better than it honestly is — smaller in magnitude here than a supervised leak, since K-Means spreads distance across the whole feature space rather than a single dominant field, but the direction and mechanism are the same.

This is a synthetic sanity check, not proof of a real production leak — an actual leak would require a genuine forbidden field (e.g. health_score, priority_score, action_type) being available pre-clustering, none of which exist among the confirmed 5 features. All leaked columns were removed. 0.2972 is the only score reported as the valid model metric going forward.

In [3]:
TABLE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
DIM = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

lane3_features = con.sql(f"""
    WITH scoped_gsc AS (
        SELECT client_hash_id, content_hash_id, report_date,
               gsc_impressions, gsc_clicks, gsc_avg_position
        FROM read_parquet('{TABLE}')
        WHERE gsc_data_available = TRUE
    ),
    gsc_agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS gsc_impressions,
            SUM(gsc_impressions * gsc_avg_position) / NULLIF(SUM(gsc_impressions), 0) AS gsc_avg_position,
            SUM(gsc_clicks) AS gsc_clicks,
            COUNT(DISTINCT report_date) AS gsc_days_present
        FROM scoped_gsc
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        g.client_hash_id, g.content_hash_id, g.gsc_impressions, g.gsc_avg_position,
        g.gsc_clicks, g.gsc_days_present,
        DATE_DIFF('day', d.content_created_date, DATE '2026-03-31') AS content_age_days,
        d.word_count
    FROM gsc_agg g
    LEFT JOIN read_parquet('{DIM}') d ON g.content_hash_id = d.content_hash_id
""").df()

lane3_features['word_count_missing_flag'] = lane3_features['word_count'].isna().astype(int)
lane3_features['word_count'] = lane3_features['word_count'].fillna(lane3_features['word_count'].median())

print("content_age_days validation:")
print("  nulls:", lane3_features['content_age_days'].isna().sum())
print("  negative:", (lane3_features['content_age_days'] < 0).sum())
print("  min/max:", lane3_features['content_age_days'].min(), lane3_features['content_age_days'].max())

print("\nword_count nulls before imputation:", (lane3_features['word_count_missing_flag']).sum(),
      f"({lane3_features['word_count_missing_flag'].mean()*100:.2f}%)")
print("Final shape:", lane3_features.shape)
lane3_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

content_age_days validation:
  nulls: 0
  negative: 0
  min/max: 0 494

word_count nulls before imputation: 55315 (31.30%)
Final shape: (176738, 9)


,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,gsc_clicks,gsc_days_present,content_age_days,word_count,word_count_missing_flag
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,4.450877,2.0,31,396,2731,1
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,2.298246,0.0,26,396,2731,1
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,5.637584,0.0,30,396,2731,1
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.906404,6.0,31,396,2731,1
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,3.950542,16.0,31,396,2475,0


In [4]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

# Honest feature set — the finalized 5 features
honest_cols = ['gsc_impressions', 'gsc_avg_position', 'gsc_clicks', 'content_age_days', 'word_count']
X_honest = lane3_features[honest_cols].copy()
X_honest['gsc_impressions'] = np.log1p(X_honest['gsc_impressions'])
X_honest['gsc_clicks'] = np.log1p(X_honest['gsc_clicks'])
X_honest_scaled = StandardScaler().fit_transform(X_honest)

km_honest = KMeans(n_clusters=5, random_state=42, n_init=10)
labels_honest = km_honest.fit_predict(X_honest_scaled)
score_honest = silhouette_score(X_honest_scaled, labels_honest, sample_size=20000, random_state=42)
print(f"HONEST silhouette score (5 legitimate features): {score_honest:.4f}")

# --- THE TRAP ---
# One-hot encode the leaked cluster labels — never ordinal, since cluster IDs
# carry no real distance relationship (cluster 3 is not "between" 2 and 4).
leak_onehot = OneHotEncoder(sparse_output=False).fit_transform(labels_honest.reshape(-1, 1))

# Primary comparison: SAME cluster assignment, leak appended to the feature space only
X_leaky_diag = np.hstack([X_honest_scaled, leak_onehot])
score_leaky_diag = silhouette_score(X_leaky_diag, labels_honest, sample_size=20000, random_state=42)
print(f"DIAGNOSTIC silhouette (leak appended, same labels): {score_leaky_diag:.4f}")

# Secondary/optional: refit K-Means on the leaky matrix, as a separate check
km_refit = KMeans(n_clusters=5, random_state=42, n_init=10)
labels_refit = km_refit.fit_predict(X_leaky_diag)
score_refit = silhouette_score(X_leaky_diag, labels_refit, sample_size=20000, random_state=42)
print(f"DIAGNOSTIC silhouette (leak appended, REFIT clustering): {score_refit:.4f}")

print(f"\nHonest -> diagnostic (same labels): {score_honest:.4f} -> {score_leaky_diag:.4f}  (+{score_leaky_diag - score_honest:.4f})")
print(f"Honest -> diagnostic (refit):        {score_honest:.4f} -> {score_refit:.4f}  (+{score_refit - score_honest:.4f})")

# --- REMOVE THE LEAK, KEEP ONLY THE HONEST MODEL ---
print(f"\nLeak removed. Reporting honest silhouette score as the valid model metric: {score_honest:.4f}")

HONEST silhouette score (5 legitimate features): 0.2948
DIAGNOSTIC silhouette (leak appended, same labels): 0.3927
DIAGNOSTIC silhouette (leak appended, REFIT clustering): 0.3927

Honest -> diagnostic (same labels): 0.2948 -> 0.3927  (+0.0979)
Honest -> diagnostic (refit):        0.2948 -> 0.3927  (+0.0979)

Leak removed. Reporting honest silhouette score as the valid model metric: 0.2948


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset can describe observable content-performance archetypes — pages that share patterns in search visibility, position, clicks, freshness, and content length. It can never establish why a page performs the way it does: not user intent quality, editorial quality, topical authority, SERP layout, backlink profile, technical SEO defects, competitor actions, algorithm-update effects, or whether a refresh would causally improve anything. Cluster names in this analysis must be treated as descriptive archetypes, never causal diagnoses or prescribed interventions.

Verified in production (real queries against the 78.8M-row warehouse, this session):



*   Coverage: the feature frame covers 176,738 / ~519,606 content items (~34%) — only content with real GSC data in March. This is not a random sample; it systematically excludes pages with no March search visibility, biasing every archetype toward search-visible content.

*   GA4/engagement sparsity: ga4_data_available IS TRUE held for only 4.2% of rows; GSC/GA4 overlap only 3.7% — confirming GA4 is the binding constraint, not an artifact of the GSC filter. This is why ga4_engaged_sessions was excluded from the core frame — the resulting archetypes describe search-side behavior only, and say almost nothing about on-page engagement for the large majority of content.

*   word_count missingness: 31.30% null in the final 176,738-row frame, handled with median imputation + a word_count_missing_flag. This is a coverage limitation, not a minor nuisance — and its cause is not verified (see below).

Named as a risk, not yet verified:

*   Client-history imbalance. Per the skill file's own warning, GSC/GA4 history depth differs by client — but this was never checked per-client in this notebook. The uniform month=2026-03 window may include clients whose tracking started partway through March, meaning some "missing" rows aren't behavioral absence but data-nonexistence. Without stratified row counts and date ranges by client_hash_id, clusters could partly reflect client tenure/instrumentation rather than stable content behavior — this is a real, open gap, not resolved by this notebook.

*   Cause of word_count/search_volume/competition missingness. Section 2 found search_volume and competition null at an identical 27.4% rate, strongly suggesting a shared upstream cause (likely content_type-linked) — but this correlation was observed, never formally tested. If word_count's 31.30% gap follows the same pattern, imputing with a global median could systematically misrepresent one content type more than others.

Structural limits by design, not by gap:


*   Single-month snapshot. March 2026 is one month of ~17 available (2025-01-27 → 2026-06-30). This can describe one observation period; it cannot show trend persistence, seasonality, or generalization to other months.

*   Window-overlap risk, inherited but not directly triggered here. The skill file notes fact_content_query_90d's 90-day window overlaps the panel's final months — a real leakage risk if this were ever combined with that table to build a past→future label. Lane 3's clustering builds no label, so this risk isn't active in this notebook, but it constrains any future extension that adds a predictive layer on top of these archetypes.

*   5-feature cap and forbidden-field exclusions. By design, main_intent, char_count, backlinks, category_count, and AI-referral signal were excluded from the core frame (candidate-but-unused, or excluded for sparsity). By rule, identifiers, availability flags, and any product-decision field were excluded to prevent circularity. Both choices make the clustering safer from leakage, but also mean it cannot explain intent, backlink authority, generation method, or prior workflow decisions — the archetypes are only as rich as five behavioral/structural signals allow.









## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.